## About Guided Cursor

Welcome to **Guided Cursor**, an AI-powered platform designed to guide you through programming problems step by step. As with all AI systems, responses may not always be perfectly accurate. You are encouraged to think critically and verify all output independently.

**Privacy.** We may collect anonymised usage data to improve the platform and support academic research into AI-assisted pedagogy. No data will be shared outside the project team. Your usage and performance will **not** be disclosed to module leaders and will have **no bearing** on your academic grades.

**Data Retention.** This platform may be taken offline at the end of the academic term, and all stored data may be permanently deleted. Please back up any materials you wish to keep in advance.

**Contact.** For any questions or concerns, please reach out to **hello@guidedcursor.studio**.

# The Classical RK4 Method and ODE Systems

**Learning objectives**

By the end of this notebook you will be able to:

- Implement the classical fourth-order Runge-Kutta (RK4) method and explain the role of each slope
- Verify fourth-order convergence via a log-log error plot
- Extend RK4 to solve systems of coupled first-order ODEs
- Apply RK4 to the Lotka-Volterra predator-prey model
- Convert a second-order ODE to a first-order system and solve the Van der Pol oscillator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ode_hints import show_hint
from ode_verify import (check_rk4, check_rk4_system, check_van_der_pol)

## Prerequisites

This notebook builds on *Euler's Method and RK2*. We reuse the same running example throughout:

$$\frac{dy}{dt} = -2y, \qquad y(0) = 1, \qquad y(t) = e^{-2t}$$

We also carry forward the Forward Euler and RK2 (Heun) solvers for comparison.

In [ ]:
# Running example: dy/dt = -2y, y(0) = 1, solution y = exp(-2t)
def f(t, y):
    """Right-hand side of the running example ODE."""
    return -2 * y

def y_exact(t):
    """Analytical solution of the running example."""
    return np.exp(-2 * t)

# Global parameters
t_span = (0, 3)
y0 = 1
t_dense = np.linspace(0, 3, 300)

# Reference implementations from the previous notebook
def forward_euler(f, t_span, y0, h):
    """Forward Euler method (first-order)."""
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = y[i] + h * f(t[i], y[i])
    return t, y

def rk2_heun(f, t_span, y0, h):
    """RK2 Heun's method (second-order)."""
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        k1 = f(t[i], y[i])
        k2 = f(t[i+1], y[i] + h * k1)
        y[i+1] = y[i] + (h / 2) * (k1 + k2)
    return t, y

## 1. The Classical Fourth-Order Runge-Kutta Method (RK4)

### From RK2 to RK4

RK2 samples the slope at **two** points in each interval and achieves $\mathcal{O}(h^2)$ accuracy. A natural idea: sample the slope at **more** points and combine them more carefully.

The **classical RK4** evaluates the slope at **four** points and combines them in a weighted average. The payoff is dramatic: the global error drops to $\mathcal{O}(h^4)$. Halving $h$ reduces the error by a factor of **16**.

### 1.1 The Four Slopes

$$k_1 = f(t_n,\; y_n)$$
$$k_2 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}\,k_1\right)$$
$$k_3 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}\,k_2\right)$$
$$k_4 = f(t_n + h,\; y_n + h\,k_3)$$

$$y_{n+1} = y_n + \frac{h}{6}\bigl(k_1 + 2k_2 + 2k_3 + k_4\bigr)$$

**What each slope represents:**

| Slope | Evaluated at | Role |
|-------|-------------|------|
| $k_1$ | Start of interval | Initial slope |
| $k_2$ | Midpoint (via $k_1$) | First midpoint correction |
| $k_3$ | Midpoint (via $k_2$) | Refined midpoint correction |
| $k_4$ | End of interval (via $k_3$) | Endpoint slope |

The midpoint slopes $k_2$, $k_3$ receive **double weight** because midpoint estimates better represent the average slope across the interval. This is the same idea behind Simpson's rule in integration.

In [ ]:
# Complete code. Just run this cell.
# Visualise the four RK4 slopes at a single step
h_vis = 0.5
t0_vis, y0_vis = 0.0, 1.0

# Compute the four slopes
k1 = f(t0_vis, y0_vis)
k2 = f(t0_vis + h_vis / 2, y0_vis + h_vis / 2 * k1)
k3 = f(t0_vis + h_vis / 2, y0_vis + h_vis / 2 * k2)
k4 = f(t0_vis + h_vis, y0_vis + h_vis * k3)
y_rk4_step = y0_vis + (h_vis / 6) * (k1 + 2*k2 + 2*k3 + k4)

fig, ax = plt.subplots(figsize=(9, 5))
t_curve = np.linspace(0, 0.7, 200)
ax.plot(t_curve, y_exact(t_curve), color="steelblue", linewidth=2,
        label="Exact $y = e^{-2t}$")

# Draw tangent lines for each slope
seg = 0.2  # half-length of each tangent segment

# k1 at (0, 1)
ts = np.linspace(t0_vis - 0.02, t0_vis + seg, 50)
ax.plot(ts, y0_vis + k1 * (ts - t0_vis), "--", color="crimson", linewidth=1.5,
        label=f"$k_1 = {k1:.1f}$ (start)")
ax.plot(t0_vis, y0_vis, "o", color="crimson", markersize=8, zorder=5)

# k2 at midpoint via k1
t_mid = t0_vis + h_vis / 2
y_mid1 = y0_vis + h_vis / 2 * k1
ts = np.linspace(t_mid - seg, t_mid + seg, 50)
ax.plot(ts, y_mid1 + k2 * (ts - t_mid), "--", color="forestgreen", linewidth=1.5,
        label=f"$k_2 = {k2:.1f}$ (midpoint via $k_1$)")
ax.plot(t_mid, y_mid1, "s", color="forestgreen", markersize=8, zorder=5)

# k3 at midpoint via k2
y_mid2 = y0_vis + h_vis / 2 * k2
ts = np.linspace(t_mid - seg, t_mid + seg, 50)
ax.plot(ts, y_mid2 + k3 * (ts - t_mid), "--", color="darkorange", linewidth=1.5,
        label=f"$k_3 = {k3:.1f}$ (midpoint via $k_2$)")
ax.plot(t_mid, y_mid2, "D", color="darkorange", markersize=8, zorder=5)

# k4 at end via k3
t_end = t0_vis + h_vis
y_end = y0_vis + h_vis * k3
ts = np.linspace(t_end - seg, t_end + seg, 50)
ax.plot(ts, y_end + k4 * (ts - t_end), "--", color="darkviolet", linewidth=1.5,
        label=f"$k_4 = {k4:.1f}$ (end via $k_3$)")
ax.plot(t_end, y_end, "^", color="darkviolet", markersize=8, zorder=5)

# Final RK4 result vs exact
ax.plot(h_vis, y_rk4_step, "*", color="black", markersize=14, zorder=6,
        label=f"RK4 result = {y_rk4_step:.4f}")
ax.plot(h_vis, y_exact(h_vis), "*", color="steelblue", markersize=10, zorder=6,
        label=f"Exact = {y_exact(h_vis):.4f}")

ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("RK4: four slopes in a single step")
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the plot above.** Starting from $(0, 1)$, $k_1 = -2.0$ is the slope of the ODE at that point (red dashed line). We follow $k_1$ halfway to get the green square at $t = 0.25$, where the slope is $k_2 = -1.0$, a gentler decline because $y$ is smaller there. We then return to $t = 0$ and step halfway again using $k_2$ instead, landing at the orange diamond (a slightly different midpoint estimate), where $k_3 = -1.5$. Finally, we take a full step using $k_3$ to reach the purple triangle at $t = 0.5$, where $k_4 = -0.5$. The weighted average $\frac{1}{6}(k_1 + 2k_2 + 2k_3 + k_4)$ gives the RK4 result (black star, 0.3750), which nearly coincides with the exact solution (blue star, 0.3679).

### 1.2 Implementation

The algorithm:

1. Create an array of time points from $t_0$ to $t_{\text{end}}$ with spacing $h$.
2. Initialise $y_0$.
3. Loop: compute $k_1, k_2, k_3, k_4$ and apply the weighted update.
4. Return both arrays.

Fill in the loop body below.

In [ ]:
def rk4(f, t_span, y0, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y = np.zeros(len(t))
    y[0] = y0

    # ===== YOUR CODE BELOW =====
    # Compute k1, k2, k3, k4 and update y.






    # ===== YOUR CODE ABOVE =====
    return t, y

In [ ]:
show_hint("rk4")

In [ ]:
check_rk4(rk4)

### 1.3 RK4 vs Euler and RK2

Let us compare all three methods on the running example with a deliberately large step size ($h = 0.5$) so the accuracy differences are clearly visible.

In [ ]:
# Complete code. Just run this cell.
h_comp = 0.5
t_fe, y_fe = forward_euler(f, t_span, y0, h_comp)
t_rk2, y_rk2 = rk2_heun(f, t_span, y0, h_comp)
t_rk4, y_rk4 = rk4(f, t_span, y0, h_comp)

plt.figure(figsize=(9, 5))
plt.plot(t_dense, y_exact(t_dense), color="steelblue", linewidth=2,
         label="Analytical: $y = e^{-2t}$")
plt.plot(t_fe, y_fe, "o-", color="crimson", markersize=5, label="Forward Euler")
plt.plot(t_rk2, y_rk2, "s-", color="forestgreen", markersize=5, label="RK2 (Heun)")
plt.plot(t_rk4, y_rk4, "D-", color="darkorange", markersize=5, label="RK4")
plt.xlabel("t")
plt.ylabel("y")
plt.title(f"Euler vs RK2 vs RK4 on $dy/dt = -2y$ (h = {h_comp})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Error at t = 1
for name, t_m, y_m in [("Euler", t_fe, y_fe),
                        ("RK2", t_rk2, y_rk2),
                        ("RK4", t_rk4, y_rk4)]:
    idx = np.argmin(np.abs(t_m - 1.0))
    err = abs(y_m[idx] - y_exact(t_m[idx]))
    print(f"{name:6s}  error at t=1: {err:.8f}")

### 1.4 Fourth-Order Convergence

RK4 has global error $\mathcal{O}(h^4)$. On a **log-log plot** of error versus $h$, the data should follow a line with slope $\approx 4$. We compare this against Euler (slope 1) and RK2 (slope 2).

In [ ]:
# Complete code. Just run this cell.
h_values = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]

errs_euler, errs_rk2, errs_rk4 = [], [], []
for hv in h_values:
    t_e, y_e = forward_euler(f, t_span, y0, hv)
    t_r2, y_r2 = rk2_heun(f, t_span, y0, hv)
    t_r4, y_r4 = rk4(f, t_span, y0, hv)
    for t_m, y_m, errs in [(t_e, y_e, errs_euler),
                           (t_r2, y_r2, errs_rk2),
                           (t_r4, y_r4, errs_rk4)]:
        idx = np.argmin(np.abs(t_m - 1.0))
        errs.append(abs(y_m[idx] - y_exact(t_m[idx])))

h_ref = np.array(h_values)

plt.figure(figsize=(8, 5))
plt.loglog(h_values, errs_euler, "o-", color="crimson", label="Forward Euler")
plt.loglog(h_values, errs_rk2, "s-", color="forestgreen", label="RK2 (Heun)")
plt.loglog(h_values, errs_rk4, "D-", color="darkorange", label="RK4")
plt.loglog(h_ref, 0.3 * h_ref, "--", color="grey", alpha=0.5, label="Slope 1")
plt.loglog(h_ref, 0.5 * h_ref**2, ":", color="grey", alpha=0.5, label="Slope 2")
plt.loglog(h_ref, 0.5 * h_ref**4, "-.", color="grey", alpha=0.5, label="Slope 4")
plt.xlabel("Step size h")
plt.ylabel("Error at t = 1")
plt.title("Convergence: Euler (order 1), RK2 (order 2), RK4 (order 4)")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. From Scalars to Systems

Every ODE we have solved so far is **scalar**: one equation for one unknown $y(t)$. Real-world problems typically involve **multiple coupled variables**, for example the position and velocity of a particle, or the populations of interacting species.

A system of $n$ first-order ODEs looks like:

$$\frac{d\mathbf{y}}{dt} = \mathbf{f}(t,\, \mathbf{y}), \qquad \mathbf{y}(t_0) = \mathbf{y}_0$$

where $\mathbf{y} = [y_1, y_2, \ldots, y_n]^T$ is a **vector**.

The key insight: **the RK4 formulae stay exactly the same**. We simply replace every scalar quantity with a NumPy array. Each $k_i$ becomes a vector of slopes, and the arithmetic (`+`, `*`) works element-wise.

### 2.1 Vectorising RK4

Adapt your scalar RK4 to handle vector-valued systems.

**Differences from the scalar version:**
- `y0` is a 1-D NumPy array (e.g. `np.array([1.0, 0.0])`)
- Results are stored in a 2-D array: `Y` with shape `(n_steps, n_variables)`
- `f(t, y)` returns a 1-D array of the same length as `y`

Fill in the loop body below.

In [ ]:
def rk4_system(f, t_span, y0, h):
    t = np.arange(t_span[0], t_span[1] + h, h)
    y0 = np.array(y0, dtype=float)
    Y = np.zeros((len(t), len(y0)))
    Y[0] = y0

    # ===== YOUR CODE BELOW =====
    # Same RK4 formula, but Y[i] is now a vector.






    # ===== YOUR CODE ABOVE =====
    return t, Y

In [ ]:
show_hint("rk4_system")

In [ ]:
check_rk4_system(rk4_system)

## 3. Application: Lotka-Volterra Predator-Prey

### 3.1 The Model

The **Lotka-Volterra equations** model the population dynamics of two species: prey ($x$, e.g. sheep) and predators ($y$, e.g. wolves).

$$\begin{cases} \dfrac{dx}{dt} = \alpha x - \beta x y \\[6pt] \dfrac{dy}{dt} = \delta x y - \gamma y \end{cases}$$

| Parameter | Meaning |
|-----------|--------|
| $\alpha$ | Prey birth rate (growth when no predators) |
| $\beta$ | Rate at which predators kill prey |
| $\delta$ | Rate at which eating prey feeds predator growth |
| $\gamma$ | Predator death rate (starvation when no prey) |

**Why do the populations cycle?** The mechanism is intuitive:
1. Abundant prey $\Rightarrow$ predators eat well $\Rightarrow$ predator population rises.
2. Too many predators $\Rightarrow$ prey are consumed faster than they reproduce $\Rightarrow$ prey decline.
3. Scarce prey $\Rightarrow$ predators starve $\Rightarrow$ predator population drops.
4. Fewer predators $\Rightarrow$ prey recover $\Rightarrow$ back to step 1.

The result is an **eternal periodic cycle**. Neither species can drive the other to extinction.

In [ ]:
# Complete code. Just run this cell.
# Lotka-Volterra parameters
alpha, beta, delta, gamma_ = 1.0, 0.1, 0.075, 1.5

def lotka_volterra(t, state):
    """Right-hand side of the Lotka-Volterra system."""
    x, y = state
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma_ * y
    return np.array([dxdt, dydt])

### 3.2 Population Dynamics

We solve the system with initial populations $x_0 = 40$ (prey) and $y_0 = 9$ (predators) and plot both populations over time. Notice how the predator peak always **lags** behind the prey peak: predators can only grow after prey become abundant.

In [ ]:
# Complete code. Just run this cell.
lv_y0 = np.array([40.0, 9.0])
t_lv, Y_lv = rk4_system(lotka_volterra, (0, 50), lv_y0, h=0.01)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_lv, Y_lv[:, 0], color="forestgreen", linewidth=2, label="Prey (sheep)")
ax.plot(t_lv, Y_lv[:, 1], color="crimson", linewidth=2, label="Predators (wolves)")
ax.set_xlabel("Time")
ax.set_ylabel("Population")
ax.set_title("Lotka-Volterra: population dynamics")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Phase Portrait

A **phase portrait** plots one variable against the other, removing time from the picture. For the Lotka-Volterra system, each trajectory forms a **closed loop** centred on the equilibrium point $(x^*, y^*) = (\gamma/\delta,\; \alpha/\beta)$. The populations orbit this point endlessly; the loop never spirals in or out.

In [ ]:
# Complete code. Just run this cell.
fig, ax = plt.subplots(figsize=(7, 6))

# Several initial conditions
for x0 in [20, 30, 40, 50]:
    t_pp, Y_pp = rk4_system(lotka_volterra, (0, 50), np.array([x0, 9.0]), h=0.01)
    ax.plot(Y_pp[:, 0], Y_pp[:, 1], linewidth=1.5, label=f"$x_0 = {x0}$")

# Mark the equilibrium
eq_x, eq_y = gamma_ / delta, alpha / beta
ax.plot(eq_x, eq_y, "k*", markersize=12,
        label=f"Equilibrium ({eq_x:.0f}, {eq_y:.0f})")

ax.set_xlabel("Prey population $x$")
ax.set_ylabel("Predator population $y$")
ax.set_title("Lotka-Volterra: phase portrait")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Second-Order ODEs: Order Reduction

### 4.1 The Idea

Many physical systems involve **second-order** ODEs (containing $y''$). Our RK4 solver handles only **first-order** systems. The standard trick: introduce a new variable for the first derivative.

Given:

$$\frac{d^2 x}{dt^2} = g\!\left(t,\, x,\, \frac{dx}{dt}\right)$$

define $v = \frac{dx}{dt}$. Then the single second-order equation becomes **two coupled first-order equations**:

$$\begin{cases} \dfrac{dx}{dt} = v \\[6pt] \dfrac{dv}{dt} = g(t,\, x,\, v) \end{cases}$$

This is a system we can solve directly with `rk4_system`. The same principle extends to any $n$th-order ODE: it becomes $n$ first-order equations.

### 4.2 The Van der Pol Oscillator

The **Van der Pol oscillator** is a nonlinear second-order ODE first studied by the Dutch physicist Balthasar van der Pol in vacuum-tube circuits. It has since found applications in modelling heartbeats, neural impulses, and many other biological rhythms:

$$\frac{d^2 x}{dt^2} - \mu(1 - x^2)\frac{dx}{dt} + x = 0$$

The parameter $\mu$ controls the **nonlinear damping**:
- When $|x|$ is small (amplitude below 1): the damping term is **negative**, so the system *pumps energy in* and oscillations grow.
- When $|x|$ is large (amplitude above 1): the damping turns **positive**, *dissipating energy* and shrinking the oscillations.

This self-regulating mechanism means every trajectory converges to a single stable periodic orbit called the **limit cycle**, regardless of the starting point.

**Order reduction.** Let $v = \frac{dx}{dt}$. Rearranging the Van der Pol equation:

$$\begin{cases} \dfrac{dx}{dt} = v \\[6pt] \dfrac{dv}{dt} = \mu(1 - x^2)\,v - x \end{cases}$$

### 4.3 Implementing the Van der Pol System

Write the right-hand side function that takes the state vector $[x, v]$ and the parameter $\mu$, and returns $[dx/dt,\; dv/dt]$ as a NumPy array.

In [ ]:
def van_der_pol(t, state, mu):
    """Right-hand side of the Van der Pol oscillator (first-order system)."""

    # ===== YOUR CODE BELOW =====




    # ===== YOUR CODE ABOVE =====

In [ ]:
show_hint("van_der_pol")

In [ ]:
check_van_der_pol(van_der_pol)

### 4.4 Time-Domain Solution

How does $\mu$ affect the oscillation? A small $\mu$ gives nearly sinusoidal motion. As $\mu$ increases, the waveform develops sharp, sudden transitions known as **relaxation oscillations**.

In [ ]:
# Complete code. Just run this cell.
vdp_y0 = np.array([2.0, 0.0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, mu_val in zip(axes, [0.5, 1.0, 5.0]):
    f_vdp = lambda t, y, mu=mu_val: van_der_pol(t, y, mu)
    t_vdp, Y_vdp = rk4_system(f_vdp, (0, 40), vdp_y0, h=0.01)
    ax.plot(t_vdp, Y_vdp[:, 0], color="steelblue", linewidth=1.5)
    ax.set_xlabel("Time")
    ax.set_ylabel("$x(t)$")
    ax.set_title(f"$\\mu = {mu_val}$")
    ax.grid(True, alpha=0.3)

plt.suptitle("Van der Pol oscillator: effect of $\\mu$", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### 4.5 Phase Portrait and the Limit Cycle

The defining feature of the Van der Pol oscillator: **all trajectories converge to the same closed orbit** in the $(x, v)$ phase plane. Starting inside the cycle, the trajectory spirals outward; starting outside, it spirals inward. This stable attractor is the **limit cycle**.

This is why the Van der Pol equation models heartbeats: no matter what perturbation the heart receives, it always returns to the same stable rhythm.

In [ ]:
# Complete code. Just run this cell.
mu_lc = 3.0
f_vdp_lc = lambda t, y: van_der_pol(t, y, mu_lc)

fig, ax = plt.subplots(figsize=(7, 6))

# Several initial conditions: some inside, some outside the limit cycle
initial_conditions = [
    np.array([0.5, 0.0]),
    np.array([4.0, 0.0]),
    np.array([0.0, 5.0]),
    np.array([-3.0, -3.0]),
    np.array([1.0, -4.0]),
]
colours = ["crimson", "forestgreen", "darkorange", "darkviolet", "saddlebrown"]

for ic, col in zip(initial_conditions, colours):
    t_pp, Y_pp = rk4_system(f_vdp_lc, (0, 50), ic, h=0.01)
    ax.plot(Y_pp[:, 0], Y_pp[:, 1], color=col, linewidth=1, alpha=0.8)
    ax.plot(ic[0], ic[1], "o", color=col, markersize=8)  # starting point

ax.set_xlabel("$x$ (displacement)")
ax.set_ylabel("$v$ (velocity)")
ax.set_title(f"Van der Pol: limit cycle ($\\mu = {mu_lc}$)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("All trajectories converge to the same closed orbit: the limit cycle.")
print("Starting inside or outside, the system always settles into the same rhythm.")

### Summary

| Method | Order | $f$ calls / step | Can handle systems? |
|--------|-------|------------------|--------------------|
| Forward Euler | 1 | 1 | Yes (with vectors) |
| RK2 (Heun) | 2 | 2 | Yes (with vectors) |
| **RK4** | **4** | **4** | **Yes (with vectors)** |

**Key takeaways:**

- RK4 is dramatically more accurate than lower-order methods for the same step size. It is the default choice for most non-stiff ODE problems.
- Any scalar ODE solver extends to systems by replacing scalars with NumPy arrays; the formulae are identical.
- A second-order ODE can always be reduced to two first-order equations by introducing the derivative as a new variable.
- The Lotka-Volterra model shows how two coupled equations produce complex cyclic behaviour.
- The Van der Pol oscillator illustrates the **limit cycle**, a stable periodic orbit that attracts all nearby trajectories, modelling the robustness of biological rhythms.